# Notebook 1 — Data Preparation (Faster R-CNN)
**Steps:** Validate labels → YOLO→COCO conversion → Class distribution

In [ ]:
# ============================================================
#  CONFIG
# ============================================================
DATASET_BASE = "dataset"   

CLASSES = [                
    "traffic light", "traffic sign", "car", "person", "bus",
    "truck", "rider", "bike", "motor", "train", "banner", "tuktuk"
]
VALID_CLASS_IDS = set(range(len(CLASSES)))

SPLITS = {
    "train": f"{DATASET_BASE}/train/train",
    "val"  : f"{DATASET_BASE}/val/val",
    "test" : f"{DATASET_BASE}/test/test",
}
print(f"Classes ({len(CLASSES)}): {CLASSES}")

## Step 1 — Install

In [ ]:
# !pip install -q torch torchvision opencv-python tqdm matplotlib pycocotools

## Step 2 — Inspect Dataset

In [ ]:
import os
print(f"{'Split':<8} {'Images':>8} {'Labels':>8} {'Ann JSON':>14}")
print("-" * 44)
for split, path in SPLITS.items():
    img_dir = os.path.join(path, "images")
    lbl_dir = os.path.join(path, "labels")
    ann     = os.path.join(path, "annotations.json")
    imgs = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
    lbls = len(os.listdir(lbl_dir)) if os.path.exists(lbl_dir) else 0
    ann_s = f" {os.path.getsize(ann)/1024**2:.1f}MB" if os.path.exists(ann) else "❌ missing"
    print(f"{split:<8} {imgs:>8,} {lbls:>8,} {ann_s:>14}")

## Step 3 — Validate & Clean Labels

In [ ]:
from pathlib import Path

DRY_RUN = True 

def is_valid_label(label_path, valid_classes):
    try:
        with open(label_path) as f:
            for i, line in enumerate(f, 1):
                parts = line.strip().split()
                if not parts: continue
                if len(parts) != 5: return False, f"line {i}: need 5 values"
                if int(parts[0]) not in valid_classes: return False, f"line {i}: bad class"
                coords = list(map(float, parts[1:]))
                if any(not (0.0 <= x <= 1.0) for x in coords): return False, f"line {i}: bad coords"
        return True, "ok"
    except Exception as e:
        return False, str(e)

print(f"{'DRY RUN' if DRY_RUN else '⚠️  LIVE — deleting files!'}\n")
for split, path in SPLITS.items():
    img_dir = Path(os.path.join(path, "images"))
    lbl_dir = Path(os.path.join(path, "labels"))
    bad = removed = 0
    for lbl in lbl_dir.glob("*.txt"):
        valid, _ = is_valid_label(lbl, VALID_CLASS_IDS)
        if not valid:
            bad += 1
            for ext in [".jpg",".jpeg",".png"]:
                img = img_dir / (lbl.stem + ext)
                if img.exists():
                    if not DRY_RUN: img.unlink()
                    removed += 1; break
            if not DRY_RUN: lbl.unlink()
    print(f"{split}: {bad} bad labels, {removed} images removed")

## Step 4 — YOLO → COCO Conversion
> Auto-skips if annotations.json already exists

In [ ]:
import os, json, cv2
from tqdm import tqdm

def yolo_to_coco(split_path, split_name, classes):
    img_dir  = os.path.join(split_path, "images")
    lbl_dir  = os.path.join(split_path, "labels")
    out_path = os.path.join(split_path, "annotations.json")
    if os.path.exists(out_path):
        print(f"⏭️  {split_name}: exists ({os.path.getsize(out_path)/1024**2:.1f}MB) — skipping")
        return
    coco = {
        "info": {"description": f"Egypt Autonomous Cars — {split_name}"},
        "categories": [{"id": i, "name": n} for i, n in enumerate(classes)],
        "images": [], "annotations": []
    }
    ann_id = 0
    img_files = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))])
    for img_id, img_file in enumerate(tqdm(img_files, desc=f"Converting {split_name}")):
        img = cv2.imread(os.path.join(img_dir, img_file))
        if img is None: continue
        h, w = img.shape[:2]
        coco["images"].append({"id": img_id, "file_name": img_file, "width": w, "height": h})
        lbl_path = os.path.join(lbl_dir, os.path.splitext(img_file)[0] + ".txt")
        if not os.path.exists(lbl_path): continue
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5: continue
                cls_id = int(parts[0])
                cx, cy, bw, bh = map(float, parts[1:])
                abs_w = bw*w; abs_h = bh*h
                abs_x = cx*w - abs_w/2; abs_y = cy*h - abs_h/2
                coco["annotations"].append({
                    "id": ann_id, "image_id": img_id, "category_id": cls_id,
                    "bbox": [abs_x, abs_y, abs_w, abs_h],
                    "area": abs_w*abs_h, "iscrowd": 0
                })
                ann_id += 1
    with open(out_path, "w") as f: json.dump(coco, f)
    print(f"{split_name}: {len(coco['images'])} images, {len(coco['annotations'])} annotations")

for name, path in SPLITS.items():
    yolo_to_coco(path, name, CLASSES)

## Step 5 — Class Distribution

In [ ]:
import os, matplotlib.pyplot as plt
from collections import Counter

lbl_dir = os.path.join(SPLITS["train"], "labels")
counts  = Counter()
for f in os.listdir(lbl_dir):
    for line in open(os.path.join(lbl_dir, f)):
        parts = line.strip().split()
        if parts: counts[int(parts[0])] += 1

names = [CLASSES[i] for i in sorted(counts)]
vals  = [counts[i]  for i in sorted(counts)]
maxv  = max(vals)

print(f"{'Class':<15} {'Count':>8} {'Imbalance':>12}")
print("-" * 38)
for n, v in zip(names, vals):
    ratio = maxv / max(v,1)
    print(f"{n:<15} {v:>8,} {ratio:>10.1f}x{'  ⚠️' if ratio>5 else ''}")

colors = ['salmon' if maxv/max(v,1)>5 else 'steelblue' for v in vals]
fig, axes = plt.subplots(1, 2, figsize=(16,5))
bars = axes[0].bar(names, vals, color=colors, alpha=0.85)
axes[0].bar_label(bars, labels=[f'{v:,}' for v in vals], fontsize=7, rotation=90, padding=3)
axes[0].set_title("Class Distribution (red=minority)"); axes[0].tick_params(axis='x', rotation=40)
axes[1].bar(names, vals, color=colors, alpha=0.85)
axes[1].set_yscale('log'); axes[1].set_title("Log Scale")
axes[1].tick_params(axis='x', rotation=40)
axes[1].axhline(maxv/5, color='red', linestyle='--', alpha=0.5, label='5x threshold')
axes[1].legend()
plt.suptitle("Training Set — Class Imbalance", fontsize=13)
plt.tight_layout()
plt.savefig("class_distribution.png", dpi=150, bbox_inches='tight')
plt.show()